In [ ]:
!pip install --upgrade biopython wordcloud transformers tf-keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 58.7 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.19.0
    Uninstalling tensorboard-2.19.0:
      Successfully uninstalled tensorboard-2.19.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.19.0
    Uninstalling tensorflow-2.19.0:
      Successfully uninstalled tensorflow-2.19.0
  Attempting uninstall: tf-keras
    Found existing installation: tf_keras 2.19.0
    Uninstalling tf_keras-2.19.0:
      Successfully uninstalled tf_keras-2.19.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstal

In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.41.2 tf-keras

Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 54.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import gc
import re
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from Bio import Entrez
from transformers import AutoTokenizer, TFAutoModel

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/NLP_Project_Final'
DATA_PATH = os.path.join(BASE_DIR, 'pubmed_cleaned_data.csv')
SAVE = os.path.join(BASE_DIR, 'Model')

if not os.path.exists(SAVE):
    os.makedirs(SAVE)

SAVE_RS = os.path.join(SAVE, 'SplitData')

if not os.path.exists(SAVE_RS):
    os.makedirs(SAVE_RS)

Mounted at /content/drive


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f">>> [LOADED] Dataset contains {len(df)} cleaned samples.")

>>> [LOADED] Dataset contains 39926 cleaned samples.


In [ ]:
null_count = df['clean_text'].isnull().sum()

if null_count > 0:
    print(f"    - Found {null_count} null rows. Removing them...")
    df = df.dropna(subset=['clean_text'])
    df['clean_text'] = df['clean_text'].astype(str)
    df = df[df['clean_text'].str.strip() != ""]
    print(f"    - Cleaned! Remaining samples: {len(df)}")
else:
    print("    - No null values found. Proceeding...")

    - Found 1 null rows. Removing them...
    - Cleaned! Remaining samples: 39925


In [ ]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['methodology'])

In [ ]:
label_names = le.classes_
print(f">>> [ENCODED] Labels: {label_names}")
for i, name in enumerate(label_names):
    print(f"    - {name} -> {i}")

>>> [ENCODED] Labels: ['Case Report' 'Meta-Analysis' 'Observational Study'
 'Randomized Controlled Trial']
    - Case Report -> 0
    - Meta-Analysis -> 1
    - Observational Study -> 2
    - Randomized Controlled Trial -> 3


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    df['clean_text'],
    df['label_encoded'],
    test_size=0.2,
    random_state=42,
    stratify=df['label_encoded']
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

train_df = pd.DataFrame({'text': X_train, 'label': y_train})
val_df = pd.DataFrame({'text': X_val, 'label': y_val})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

train_df.to_csv(os.path.join(SAVE_RS, 'train_data.csv'), index=False, encoding='utf-8-sig')
val_df.to_csv(os.path.join(SAVE_RS, 'val_data.csv'), index=False, encoding='utf-8-sig')
test_df.to_csv(os.path.join(SAVE_RS, 'test_data.csv'), index=False, encoding='utf-8-sig')

print(f">>> [SAVED] 3 data splits saved to: {SAVE_RS}")
print(f"    - Train: {len(train_df)} samples")
print(f"    - Val:   {len(val_df)} samples")
print(f"    - Test:  {len(test_df)} samples")

>>> [SAVED] 3 data splits saved to: /content/drive/MyDrive/NLP_Project_Final/Model/SplitData
    - Train: 31940 samples
    - Val:   3992 samples
    - Test:  3993 samples
